In [0]:
# Databricks Auto Loader - Streaming Transformations

# ============================================================
# Step 01 - Setup
# ============================================================

catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/streaming_transformations"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

table_name = f"{catalog}.{schema}.orders_transformed"

dbutils.fs.mkdirs(landing)

print("Streaming transformation environment created.")




In [0]:
# ============================================================
# Step 02 - Create sample JSON files
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_01.json",
    '{"order_id": 1, "customer": "Jhansi", "amount": 250}\n'
    '{"order_id": 2, "customer": "Sathish", "amount": 120}\n'
    '{"order_id": 3, "customer": "Jhashvin", "amount": 890}\n',
    True
)

print("Sample JSON file created.")

In [0]:

# ============================================================
# Step 03 - Read data using Auto Loader
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)


In [0]:

# ============================================================
# Step 04 - Apply streaming transformations
# ============================================================

transformed_stream = (
    orders_stream
    .withColumnRenamed("order_id", "sales_order_id")
    .withColumnRenamed("customer", "customer_name")
    .withColumnRenamed("amount", "order_amount")
)


In [0]:

# ============================================================
# Step 05 - Add a derived column
# ============================================================

from pyspark.sql.functions import col, when

transformed_stream = (
    transformed_stream
    .withColumn(
        "order_category",
        when(col("order_amount") >= 500, "High Value")
        .otherwise("Regular")
    )
)



In [0]:

# ============================================================
# Step 06 - Write transformed data to Delta
# ============================================================

query = (
    transformed_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("Streaming transformation completed.")


In [0]:

# ============================================================
# Step 07 - Check transformed data
# ============================================================

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY sales_order_id
    """)
)

In [0]:
# ============================================================
# Step 08 - Add another incoming file
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_02.json",
    '{"order_id": 4, "customer": "Seeman", "amount": 650}\n'
    '{"order_id": 5, "customer": "Adiyavan", "amount": 180}\n',
    True
)

print("New JSON file added.")

In [0]:
# ============================================================
# Step 09 - Process the new file
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing)
)

transformed_stream = (
    orders_stream
    .withColumnRenamed("order_id", "sales_order_id")
    .withColumnRenamed("customer", "customer_name")
    .withColumnRenamed("amount", "order_amount")
    .withColumn(
        "order_category",
        when(col("order_amount") >= 500, "High Value")
        .otherwise("Regular")
    )
)

query = (
    transformed_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table_name)
)

print("New file processed.")



In [0]:

# ============================================================
# Step 10 - Final result
# ============================================================

display(
    spark.sql(f"""
        SELECT *
        FROM {table_name}
        ORDER BY sales_order_id
    """)
)